# OmniBird — **Point-cloud-SSL-aligned architecture** on CIFAR10-DVS

This notebook is the "v3" architecture that brings OmniBird in line with the
Point-MAE / Point-BERT / Point-M2AE / i-JEPA conventions for sparse-input SSL:

| Component | What it is |
|-----------|------------|
| **Encoder** | per-event tokens (8192), **hierarchical** attention per layer: within-group dense (G=16) + between-group dense over N/G=512 group summaries; centroid positional residual at every layer |
| **Predictor** | **Perceiver-style** — only N_q=4 *group-level* queries (one per target block centroid) cross-attend to the long context-encoder sequence |
| **Target** | **Per-group** features — for each of 4 target blocks, mean-pool the target encoder's per-event features inside that block → 4 group summary tokens |
| **JEPA loss** | smooth-L1 / cosine over **(B, 4, D)** — 4 predictions per sample, matching i-JEPA / Point-MAE patch-level convention |

Compute vs the previous per-event-predictor setup: ~25× cheaper per step.
Memory: ~7000× smaller predictor score matrix.
Architecture preserves per-event encoder features for downstream per-event tasks.


## 1. Setup

In [ ]:
import os, sys, math, time, copy, ssl, glob
sys.path.insert(0, os.path.abspath('..'))
ssl._create_default_https_context = ssl._create_unverified_context

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import matplotlib.pyplot as plt

from omnibird import (
    OmniBirdConfig, OmniBirdEncoder, PerceiverPredictor,
    orderings_from_batch, TargetCenter, ema_update, make_momentum_schedule,
    gather_target_features, gather_group_target_features,
    jepa_loss, diag_dict, fmt_diag,
    quick_probe, save_atomic, ensure_dir, short_params, count_params,
    build_loaders,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0); np.random.seed(0)

N_GPUS_REQUESTED = 4
N_GPUS = min(N_GPUS_REQUESTED, torch.cuda.device_count())
USE_DP = (DEVICE == "cuda") and (N_GPUS > 1)
print(f"GPUs visible = {torch.cuda.device_count()}  using = {N_GPUS}  DataParallel = {USE_DP}")

DATA_LAYOUT = "omnibird"
DATA_ROOT   = "../data/cifar10_dvs_omnibird"
TRAIN_FRAC  = 0.8

cfg = OmniBirdConfig()
cfg.coord_dim       = 3
cfg.signal_dim      = 2                # one-hot ON/OFF
cfg.n_events_total  = 0                # use all real events; pad to n_events_max
cfg.n_events_max    = 8192
cfg.n_ctx           = 3000             # ~37% of pool as context
cfg.n_tgt_per_block = 384              # ~19% of pool as targets (4 blocks)
cfg.n_pred_blocks   = 4
cfg.n_tgt           = cfg.n_pred_blocks * cfg.n_tgt_per_block   # = 1536
cfg.side            = 64
cfg.context_target_margin = 0.05

# ── HIERARCHICAL ENCODER + PERCEIVER PREDICTOR ─────────────────────────
cfg.attention_type   = "grouped_hierarchical"   # within-group + between-group per layer
cfg.group_size       = 16                       # G — events per attention window
cfg.pool             = "mean"                    # how to summarize each group ("mean"|"max")
cfg.use_centroid_pos = True                      # γ(group_centroid) residual at every layer

# Anti-degradation (carry over)
cfg.loss_type      = "cosine"
cfg.use_centering  = False
cfg.ema_end        = 0.9999
cfg.probe_patience = 5

# Optim
cfg.epochs         = 100
cfg.batch_size     = 64                # 64 / 4 = 16 per GPU
cfg.lr             = 8e-4              # linear-scaled
cfg.probe_interval = 10
cfg.probe_epochs   = 3
cfg.log_every      = 25
cfg.ckpt_dir       = "./checkpoints_omnibird_cifar10_dvs_hier"
ensure_dir(cfg.ckpt_dir)

print(f"events/sample = {cfg.n_events_max}  n_ctx = {cfg.n_ctx}  n_tgt = {cfg.n_tgt}")
print(f"attention={cfg.attention_type}  G={cfg.group_size}  pool={cfg.pool}  centroid_pos={cfg.use_centroid_pos}")


## 2. Dataset (same as the baseline CIFAR10-DVS notebook)

In [ ]:
class CIFAR10DVSFromClips:
    # Reads OmniBird per-clip layout (clip_*/events_0.npy + label_0.txt).
    def __init__(self, root, sensor_hw=(128, 128), n_classes=10):
        self.root = root
        self.h, self.w = sensor_hw
        self.n_classes = n_classes
        self.clips = sorted(glob.glob(os.path.join(root, "clip_*")))
        if not self.clips:
            raise RuntimeError(
                f"No clip_* dirs found in {root}.\n"
                f"  cwd: {os.getcwd()}\n"
                f"  resolved: {os.path.abspath(root)}\n"
                f"Re-run the converter (datasets/download.py)."
            )

    def __len__(self):
        return len(self.clips)

    def __getitem__(self, idx):
        clip = self.clips[idx]
        ev = np.load(os.path.join(clip, "events_0.npy")).astype(np.float32)
        if ev.shape[0] == 0:
            ev = np.zeros((1, 4), dtype=np.float32)
        x = ev[:, 0] / max(self.w - 1, 1) * 2.0 - 1.0
        y = ev[:, 1] / max(self.h - 1, 1) * 2.0 - 1.0
        t_raw = ev[:, 2]
        t = (t_raw - t_raw.min()) / max(t_raw.max() - t_raw.min(), 1.0) * 2.0 - 1.0
        p = ev[:, 3]
        if p.max() <= 1.0 and p.min() >= 0.0:
            p = p * 2.0 - 1.0
        out = np.stack([x, y, t, p], axis=1).astype(np.float32)
        with open(os.path.join(clip, "label_0.txt")) as f:
            label = int(f.read().strip())
        return out, label


base = CIFAR10DVSFromClips(DATA_ROOT, sensor_hw=(128, 128))
n = len(base)
rng = np.random.RandomState(0)
perm = rng.permutation(n)
n_train = int(n * TRAIN_FRAC)
train_idx, test_idx = perm[:n_train], perm[n_train:]

class _Subset:
    def __init__(self, base, idx): self.base = base; self.idx = idx
    def __len__(self): return len(self.idx)
    def __getitem__(self, i): return self.base[int(self.idx[i])]

base_train = _Subset(base, train_idx)
base_test  = _Subset(base, test_idx)
print(f"loaded CIFAR10-DVS: train={len(base_train)}  test={len(base_test)}")

train_loader, train_eval_loader, test_loader = build_loaders(base_train, base_test, cfg, num_workers=2)
print(f"loaders: train={len(train_loader)}  train_eval={len(train_eval_loader)}  test={len(test_loader)}")


## 3. Models — hierarchical encoder + perceiver predictor

In [ ]:
context_encoder = OmniBirdEncoder(
    d_model=cfg.d_model, n_layers=cfg.n_layers_enc,
    n_heads=cfg.n_heads, dim_head=cfg.dim_head,
    block_size=cfg.block_size, window=cfg.window,
    n_random=cfg.n_random, n_global=cfg.n_global,
    ffn_mult=cfg.ffn_mult,
    signal_dim=cfg.signal_dim, coord_dim=cfg.coord_dim,
    fourier_dim=cfg.fourier_dim, fourier_scale=cfg.fourier_scale,
    serial_orders=cfg.serial_orders,
    reinject_pos=cfg.reinject_pos,
    attention_type=cfg.attention_type,
    group_size=cfg.group_size,
    pool=cfg.pool,
    use_centroid_pos=cfg.use_centroid_pos,
).to(DEVICE)

target_encoder = copy.deepcopy(context_encoder).to(DEVICE)
for q in target_encoder.parameters():
    q.requires_grad_(False)

# NEW: PerceiverPredictor — 4 group-level queries cross-attend to context
predictor = PerceiverPredictor(
    d_model=cfg.d_model, d_pred=cfg.d_pred,
    n_layers=cfg.n_layers_pred,
    n_heads=cfg.n_heads_pred, dim_head=cfg.dim_head_pred,
    coord_dim=cfg.coord_dim,
    fourier_dim=cfg.fourier_dim, fourier_scale=cfg.fourier_scale,
    ffn_mult=cfg.ffn_mult, pos_symmetric=cfg.predictor_pos_symmetric,
).to(DEVICE)

target_center = TargetCenter(cfg.d_model, momentum=0.9).to(DEVICE)

if USE_DP:
    device_ids = list(range(N_GPUS))
    context_encoder = nn.DataParallel(context_encoder, device_ids=device_ids)
    target_encoder  = nn.DataParallel(target_encoder,  device_ids=device_ids)
    predictor       = nn.DataParallel(predictor,       device_ids=device_ids)
    print(f"wrapped in DataParallel on {device_ids}")

def _unwrap(m):
    return m.module if isinstance(m, nn.DataParallel) else m

print(f"context_encoder: {short_params(_unwrap(context_encoder))}  "
      f"predictor (perceiver): {short_params(_unwrap(predictor))}")


## 4. Optimizer + schedules + resume

In [ ]:
params = list(_unwrap(context_encoder).parameters()) + list(_unwrap(predictor).parameters())
optimizer = AdamW(params, lr=cfg.lr, weight_decay=cfg.weight_decay)
total_steps = cfg.epochs * len(train_loader)
scheduler = CosineAnnealingLR(optimizer, T_max=total_steps)
momentum_gen = make_momentum_schedule(cfg.ema_start, cfg.ema_end, total_steps)

PBB_LAST       = os.path.join(cfg.ckpt_dir, "omnibird_last.pt")
PBB_BEST       = os.path.join(cfg.ckpt_dir, "omnibird_best.pt")
PBB_PROBE_BEST = os.path.join(cfg.ckpt_dir, "omnibird_probe_best.pt")

RESUME = True
history = {"loss": [], "diag_steps": [], "diag_log": [], "probe_accs": []}
start_epoch, best_loss, global_step = 1, float("inf"), 0
best_probe_acc, probe_no_improve = -1.0, 0
m = cfg.ema_start

if RESUME and os.path.exists(PBB_LAST):
    s = torch.load(PBB_LAST, map_location=DEVICE, weights_only=False)
    _unwrap(context_encoder).load_state_dict(s["context_encoder"])
    _unwrap(target_encoder).load_state_dict(s["target_encoder"])
    _unwrap(predictor).load_state_dict(s["predictor"])
    target_center.load_state_dict(s["center"])
    optimizer.load_state_dict(s["opt"]); scheduler.load_state_dict(s["sched"])
    history = s.get("history", history)
    global_step = s.get("global_step", 0)
    best_loss = s.get("best_loss", float("inf"))
    best_probe_acc = s.get("best_probe_acc", -1.0)
    probe_no_improve = s.get("probe_no_improve", 0)
    start_epoch = s["epoch"] + 1
    for _ in range(global_step):
        try: m = next(momentum_gen)
        except StopIteration: m = cfg.ema_end
    print(f"resumed @ ep {s['epoch']}, step {global_step}, best_probe_acc={best_probe_acc:.4f}")
else:
    print("starting fresh.")


## 5. Probe wrapper

In [ ]:
def move_orderings(ords, device):
    return {k: {kk: vv.to(device) for kk, vv in v.items()} for k, v in ords.items()}

def probe_now(num_epochs=cfg.probe_epochs, lr=1e-3):
    return quick_probe(
        _unwrap(context_encoder), train_eval_loader, test_loader,
        d_model=cfg.d_model, n_classes=10,
        num_epochs=num_epochs, lr=lr, weight_decay=1e-4, device=DEVICE,
        use_attn_pool=cfg.probe_use_attn_pool,
    )


## 6. Training loop — group-level prediction

Key differences from the per-event loop:

```python
# Old (per-event, expensive):
h_pred = predictor(g_ctx, tgt_coords, ctx_coords=ctx_c)        # (B, 1536, D)
h_tgt  = gather_target_features(g_tgt, tgt_pp)                  # (B, 1536, D)
loss   = jepa_loss(h_pred, h_tgt)                               # 1536 predictions

# New (group-level, Point-MAE convention):
group_centroids = tgt_coords.view(B, 4, n_tgt_per_block, 3).mean(dim=2)   # (B, 4, 3)
h_pred = predictor(g_ctx, group_centroids, ctx_coords=ctx_c)              # (B, 4, D)
h_tgt  = gather_group_target_features(g_tgt, tgt_pp, 4, n_tgt_per_block)  # (B, 4, D)
loss   = jepa_loss(h_pred, h_tgt)                                          # 4 predictions
```

The encoder still produces per-event features for downstream tasks.


In [ ]:
def move_orderings(ords, device):
    return {k: {kk: vv.to(device) for kk, vv in v.items()} for k, v in ords.items()}


epoch = start_epoch - 1
try:
    for epoch in range(start_epoch, cfg.epochs + 1):
        context_encoder.train(); predictor.train()
        epoch_loss, steps = 0.0, 0
        t0 = time.time()

        for batch in train_loader:
            ctx_s  = batch["ctx_signal"].to(DEVICE)
            ctx_c  = batch["ctx_coords"].to(DEVICE)
            pool_s = batch["pool_signal"].to(DEVICE)
            pool_c = batch["pool_coords"].to(DEVICE)
            tgt_c  = batch["tgt_coords"].to(DEVICE)
            tgt_pp = batch["tgt_pool_pos"].to(DEVICE)
            pool_kpm = batch["pool_kpm"].to(DEVICE) if "pool_kpm" in batch else None
            ctx_o  = move_orderings(orderings_from_batch(batch, "ctx"),  DEVICE)
            pool_o = move_orderings(orderings_from_batch(batch, "pool"), DEVICE)

            with torch.no_grad():
                g_tgt = target_encoder(pool_s, pool_c, pool_o, key_padding_mask=pool_kpm)
                # ── Point-MAE-style: per-group target features ─────────
                h_tgt_raw = gather_group_target_features(
                    g_tgt, tgt_pp,
                    n_pred_blocks=cfg.n_pred_blocks,
                    n_tgt_per_block=cfg.n_tgt_per_block,
                    pool=cfg.pool,
                )                                            # (B, 4, D)
                if cfg.use_centering:
                    target_center.update(h_tgt_raw)
                    h_tgt = F.layer_norm(target_center(h_tgt_raw), (h_tgt_raw.size(-1),))
                else:
                    h_tgt = F.layer_norm(h_tgt_raw, (h_tgt_raw.size(-1),))

            # ── Compute target-block centroids as predictor queries ──
            B = tgt_c.size(0)
            n_pred = cfg.n_pred_blocks
            n_tpb  = cfg.n_tgt_per_block
            group_query_coords = tgt_c.view(B, n_pred, n_tpb, cfg.coord_dim).mean(dim=2)   # (B, 4, 3)

            g_ctx  = context_encoder(ctx_s, ctx_c, ctx_o)
            h_pred = predictor(g_ctx, group_query_coords,
                                ctx_coords=ctx_c if cfg.predictor_pos_symmetric else None)
                                                            # (B, 4, D)

            loss = jepa_loss(h_pred, h_tgt, loss_type=cfg.loss_type)
            optimizer.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            optimizer.step(); scheduler.step()

            try: m = next(momentum_gen)
            except StopIteration: m = cfg.ema_end
            ema_update(_unwrap(target_encoder), _unwrap(context_encoder), m)

            global_step += 1; epoch_loss += loss.item(); steps += 1
            if global_step % cfg.log_every == 0:
                d = diag_dict(loss, h_pred, h_tgt, g_ctx, target_center)
                print(fmt_diag(d, global_step, epoch, scheduler.get_last_lr()[0], m))
                history["diag_steps"].append(global_step); history["diag_log"].append(d)

        avg = epoch_loss / max(steps, 1)
        history["loss"].append(avg)
        improved = avg < best_loss
        if improved: best_loss = avg

        ckpt_state = {
            "epoch": epoch, "cfg": cfg.__dict__,
            "context_encoder": _unwrap(context_encoder).state_dict(),
            "target_encoder":  _unwrap(target_encoder).state_dict(),
            "predictor":       _unwrap(predictor).state_dict(),
            "center":          target_center.state_dict(),
            "opt": optimizer.state_dict(), "sched": scheduler.state_dict(),
            "global_step": global_step, "best_loss": best_loss,
            "best_probe_acc": best_probe_acc, "probe_no_improve": probe_no_improve,
            "history": history,
        }
        if improved: save_atomic(ckpt_state, PBB_BEST)
        save_atomic(ckpt_state, PBB_LAST)
        print(f"=== ep {epoch:03d}/{cfg.epochs}  avg_loss={avg:.4f}  m={m:.4f}  {time.time()-t0:.1f}s"
              + ("  ★ new best (saved omnibird_best.pt)" if improved else ""))

        if cfg.probe_interval > 0 and epoch % cfg.probe_interval == 0:
            t_probe = time.time()
            print(f"  → Running {cfg.probe_epochs}-epoch linear probe at epoch {epoch}...")
            acc = probe_now()
            history.setdefault("probe_accs", []).append((epoch, acc))
            print(f"  [probe ep {epoch:03d}]  test_acc = {acc:.4f}  ({time.time()-t_probe:.1f}s)")
            if acc > best_probe_acc:
                best_probe_acc = acc; probe_no_improve = 0
                ckpt_state["best_probe_acc"]   = best_probe_acc
                ckpt_state["probe_no_improve"] = probe_no_improve
                ckpt_state["history"]          = history
                save_atomic(ckpt_state, PBB_PROBE_BEST)
                print(f"  ★ new best probe acc → {best_probe_acc:.4f}  (saved omnibird_probe_best.pt)")
            else:
                probe_no_improve += 1
                pp = cfg.probe_patience if cfg.probe_patience > 0 else "∞"
                print(f"  no probe improvement ({probe_no_improve}/{pp})  best so far = {best_probe_acc:.4f}")
            if cfg.probe_patience > 0 and probe_no_improve >= cfg.probe_patience:
                print(f"\nEarly stopping at epoch {epoch}: best probe acc = {best_probe_acc:.4f}")
                break

    print("\nDone.")
except KeyboardInterrupt:
    print(f"\nInterrupted at epoch {epoch}.  Checkpoints in {cfg.ckpt_dir}.")


## 7. Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
if history["loss"]:
    axes[0].plot(range(1, len(history["loss"])+1), history["loss"], lw=1.5)
    axes[0].set_xlabel("epoch"); axes[0].set_ylabel("avg loss"); axes[0].set_title("JEPA loss"); axes[0].grid(alpha=0.3)
if history["diag_log"]:
    steps = history["diag_steps"]
    cos_mean = [d["cos_mean"] for d in history["diag_log"]]
    axes[1].plot(steps, cos_mean, color='C2'); axes[1].set_ylim(-0.1, 1.05)
    axes[1].set_xlabel("step"); axes[1].set_title("cos(h_pred, h_tgt) — per-group")
    axes[1].grid(alpha=0.3)
if history.get("probe_accs"):
    eps, accs = zip(*history["probe_accs"])
    axes[2].plot(eps, [a*100 for a in accs], 'o-', color='C3', markersize=6)
    axes[2].set_ylim(0, 100)
    axes[2].set_xlabel("epoch"); axes[2].set_ylabel("probe acc (%)")
    axes[2].set_title(f"probe acc (best = {max(accs)*100:.2f}%)"); axes[2].grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 8. Final probe (longer run on the probe-best checkpoint)

In [ ]:
LOAD_PROBE_BEST = True
PROBE_EPOCHS_LONG = 10
PROBE_LR = 1e-3

if LOAD_PROBE_BEST and os.path.exists(PBB_PROBE_BEST):
    s = torch.load(PBB_PROBE_BEST, map_location=DEVICE, weights_only=False)
    _unwrap(context_encoder).load_state_dict(s["context_encoder"])
    _unwrap(target_encoder).load_state_dict(s["target_encoder"])
    print(f"loaded encoders from {PBB_PROBE_BEST}  (ep {s.get('epoch','?')}, best probe={s.get('best_probe_acc', float('nan')):.4f})")

print(f"running {PROBE_EPOCHS_LONG}-epoch probe ...")
t0 = time.time()
acc = probe_now(num_epochs=PROBE_EPOCHS_LONG, lr=PROBE_LR)
print(f"\nfinal probe acc(CIFAR10-DVS) = {acc*100:.2f}%   ({time.time()-t0:.1f}s)")

if history.get("probe_accs"):
    eps, accs = zip(*history["probe_accs"])
    print(f"\nduring-training trace ({len(accs)} probes):")
    for e, a in history["probe_accs"]:
        marker = "  <-- best" if a == max(accs) else ""
        print(f"  ep {e:3d}:  {a*100:5.2f}%{marker}")
    print(f"\nfinal vs best in-training: {acc*100:.2f}% vs {max(accs)*100:.2f}% "
          f"({(acc - max(accs))*100:+.2f} pp)")
